In [11]:
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

WORD_PATTERN = re.compile(r"\b[\w\']+\b")
INPUT_FOLDER = Path("input")
OUTPUT_FOLDER = Path("output")
CHUNK_SIZE = 1000
NUM_REDUCERS = 2


In [12]:
def read_and_split(input_folder, chunk_size=1000):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    files = sorted(Path(input_folder).glob('*.txt'))
    if not files:
        raise FileNotFoundError(f"No .txt files found in {Path(input_folder).resolve()}")

    splits = []
    split_id = 0
    for file_path in files:
        text = file_path.read_text(encoding='utf-8', errors='replace')
        words = text.split()
        for start in range(0, len(words), chunk_size):
            chunk = ' '.join(words[start:start + chunk_size])
            splits.append((file_path.name, split_id, chunk))
            split_id += 1
    return splits

splits = read_and_split(INPUT_FOLDER, CHUNK_SIZE)
print(f"Files found: {len(list(INPUT_FOLDER.glob('*.txt')))}")
print(f"Splits created: {len(splits)}")
splits[:3]

Files found: 5
Splits created: 5


[('book_01.txt',
  0,
  'Distributed systems process large volumes of information across many machines. Map tasks split the input and emit intermediate key value pairs. Reduce tasks aggregate all values that share the same key. Students can test tokenization normalization and counting accuracy. Reliable analytics requires careful handling of noise and duplicate terms.'),
 ('book_02.txt',
  1,
  'Big data pipelines often begin with log ingestion and parsing. A mapper can read each line and extract meaningful words. Another mapper can emit composite keys for advanced analysis. Reducers combine partial counts from many independent mapper outputs. This sample file intentionally repeats words words words for testing.'),
 ('book_03.txt',
  2,
  'Cloud storage enables scalable processing for educational projects. MapReduce supports parallel execution when data is partitioned correctly. Intermediate results are shuffled and sorted before reduction starts. Engineers validate correctness by comp

In [13]:
def map_split(split):
    filename, split_id, text = split
    pairs = [(word, 1) for word in WORD_PATTERN.findall(text.lower())]
    return filename, split_id, pairs

mapped_results = [map_split(split) for split in splits]
total_pairs = sum(len(pairs) for _, _, pairs in mapped_results)
print(f"Mapped pairs emitted: {total_pairs}")
mapped_results[0]

Mapped pairs emitted: 235


('book_01.txt',
 0,
 [('distributed', 1),
  ('systems', 1),
  ('process', 1),
  ('large', 1),
  ('volumes', 1),
  ('of', 1),
  ('information', 1),
  ('across', 1),
  ('many', 1),
  ('machines', 1),
  ('map', 1),
  ('tasks', 1),
  ('split', 1),
  ('the', 1),
  ('input', 1),
  ('and', 1),
  ('emit', 1),
  ('intermediate', 1),
  ('key', 1),
  ('value', 1),
  ('pairs', 1),
  ('reduce', 1),
  ('tasks', 1),
  ('aggregate', 1),
  ('all', 1),
  ('values', 1),
  ('that', 1),
  ('share', 1),
  ('the', 1),
  ('same', 1),
  ('key', 1),
  ('students', 1),
  ('can', 1),
  ('test', 1),
  ('tokenization', 1),
  ('normalization', 1),
  ('and', 1),
  ('counting', 1),
  ('accuracy', 1),
  ('reliable', 1),
  ('analytics', 1),
  ('requires', 1),
  ('careful', 1),
  ('handling', 1),
  ('of', 1),
  ('noise', 1),
  ('and', 1),
  ('duplicate', 1),
  ('terms', 1)])

In [14]:
def run_mappers_in_parallel(splits, max_workers=None):
    if not splits:
        return []
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(map_split, split) for split in splits]
        for future in as_completed(futures):
            results.append(future.result())
    return sorted(results, key=lambda result: result[1])

parallel_mapped_results = run_mappers_in_parallel(splits)
print(f"Mapper tasks completed: {len(parallel_mapped_results)}")
print(f"Total mapped pairs: {sum(len(pairs) for _, _, pairs in parallel_mapped_results)}")

Mapper tasks completed: 5
Total mapped pairs: 235


In [15]:
def combine_and_partition(mapped_results, num_reducers=2):
    if num_reducers <= 0:
        raise ValueError("num_reducers must be greater than zero.")
    partitions = defaultdict(list)
    for filename, split_id, pairs in mapped_results:
        local_counts = Counter(word for word, _ in pairs)
        for word, count in local_counts.items():
            # Stable partition function; avoids Python's randomized hash().
            reducer_id = sum(word.encode('utf-8')) % num_reducers
            partitions[reducer_id].append((word, count))
    for reducer_id in range(num_reducers):
        partitions.setdefault(reducer_id, [])
    return partitions

partitions = combine_and_partition(parallel_mapped_results, NUM_REDUCERS)
for reducer_id in range(NUM_REDUCERS):
    print(f"Reducer {reducer_id}: {len(partitions[reducer_id])} combined records")

Reducer 0: 107 combined records
Reducer 1: 108 combined records


In [16]:
def shuffle_sort_reduce(partitions, output_folder, num_reducers=2):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    final_counts = {}
    for reducer_id in range(num_reducers):
        grouped = defaultdict(list)
        for word, count in partitions.get(reducer_id, []):
            grouped[word].append(count)
        reduced = {word: sum(counts) for word, counts in grouped.items()}
        final_counts.update(reduced)
        output_file = output_folder / f'part-{reducer_id:05d}.txt'
        with output_file.open('w', encoding='utf-8') as file:
            for word in sorted(reduced):
                file.write(f'{word}\t{reduced[word]}\n')
    return final_counts

final_counts = shuffle_sort_reduce(partitions, OUTPUT_FOLDER, NUM_REDUCERS)
print(f"Unique words: {len(final_counts)}")
print(f"Output folder: {OUTPUT_FOLDER.resolve()}")

Unique words: 182
Output folder: D:\CS\Semster 7\Big data analytics\Bonus\output


In [17]:
for word in sorted(final_counts):
    print(f'{word}\t{final_counts[word]}')

a	1
accuracy	1
across	1
advanced	1
aggregate	1
aggregation	1
all	1
an	1
analysis	2
analytics	2
and	12
another	1
are	1
before	2
begin	1
behavior	1
big	1
by	1
can	5
careful	1
cases	1
casing	1
cause	1
cloud	1
combine	1
combiner	1
combiners	1
comparing	1
composite	1
confidence	1
contain	1
corpora	1
correctly	1
correctness	1
counting	1
counts	1
cross	1
custom	1
data	2
datasets	1
debug	1
determinism	1
distributed	1
documents	1
duplicate	1
each	2
edge	1
educational	1
emit	2
empty	1
enables	1
engineers	1
execution	1
expected	1
experiments	1
extract	1
extraction	1
file	2
files	1
for	4
framework	1
frequency	1
from	1
handling	1
helps	1
improve	1
in	1
include	1
includes	2
independent	1
index	1
indexing	1
information	1
ingestion	1
input	1
intentionally	1
intermediate	2
inverted	1
involve	1
is	1
it	1
key	2
keys	2
keyword	1
language	1
large	1
letter	1
line	1
lines	1
log	1
logic	1
machines	1
many	2
map	2
mapper	3
mapreduce	1
maps	1
may	1
meaningful	1
mixed	1
multiple	1
network	1
noise	1
normalization	

In [18]:
for output_file in sorted(OUTPUT_FOLDER.glob('part-*.txt')):
    print(f'\n--- {output_file.name} ---')
    print(output_file.read_text(encoding='utf-8'), end='')


--- part-00000.txt ---
advanced	1
aggregation	1
analysis	2
analytics	2
are	1
behavior	1
big	1
can	5
careful	1
combiners	1
comparing	1
confidence	1
contain	1
corpora	1
counts	1
cross	1
data	2
documents	1
enables	1
engineers	1
execution	1
expected	1
experiments	1
file	2
framework	1
frequency	1
from	1
helps	1
improve	1
include	1
independent	1
index	1
indexing	1
information	1
ingestion	1
input	1
intentionally	1
is	1
keys	2
language	1
letter	1
line	1
log	1
logic	1
machines	1
map	2
mapreduce	1
meaningful	1
multiple	1
network	1
noise	1
now	1
numbers	1
observed	1
often	1
outputs	1
own	1
parsing	1
partitioners	1
performance	1
plain	1
projects	1
punctuation	1
read	1
reduce	2
reducer	3
reliable	1
remove	1
repeats	1
repository	1
requires	1
results	1
same	1
sample	1
split	1
stragglers	1
students	3
supports	1
systems	1
tasks	2
term	1
test	1
testing	2
their	1
this	2
tokens	1
validate	1
values	1
when	1
with	3
workloads	1

--- part-00001.txt ---
a	1
accuracy	1
across	1
aggregate	1
all	1
an	1
and	12
an